In [ ]:
import os
import numpy as np
import tensorflow as tf
from tqdm import tqdm
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# -------------------------------
# Data Loading and Preprocessing
# -------------------------------

DATA_DIR_FAKE = "C:/Users/M2-Winterfell/Downloads/pulse2pulse_150k/from_006_chkp_2500_150k"
DATA_DIR_REAL = "C:/Users/M2-Winterfell/Downloads/GAN-models-for-Bio-Authentication-through-ECG-signals/datasets/real_ecgs"

def load_lead_I_from_asc(file_path):
    try:
        ecg_data = np.loadtxt(file_path)
        lead_I = ecg_data[:, 0]  # Extract first column (Lead I)
        return lead_I
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def load_exactly_n_lead_I_from_directory(data_dir, n=2000):
    all_lead_I = []
    i = 0
    with tqdm(total=n, desc=f"Loading Lead I from {os.path.basename(data_dir)}", unit='file') as pbar:
        while len(all_lead_I) < n:
            file_name = f"{i}.asc"
            file_path = os.path.join(data_dir, file_name)
            if os.path.exists(file_path):
                lead_I = load_lead_I_from_asc(file_path)
                if lead_I is not None:
                    all_lead_I.append(lead_I)
                    pbar.update(1)
            i += 1
    return np.array(all_lead_I)

# Load data from both directories
lead_I_real = load_exactly_n_lead_I_from_directory(DATA_DIR_REAL, n=2000)
lead_I_fake = load_exactly_n_lead_I_from_directory(DATA_DIR_FAKE, n=2000)

if lead_I_fake.size > 0 and lead_I_real.size > 0:
    print(f"Total Lead I records loaded from real data: {lead_I_real.shape[0]}")
    print(f"Total Lead I records loaded from fake data: {lead_I_fake.shape[0]}")
else:
    print("No data was loaded.")

def min_max_normalize(data):
    min_val = np.min(data)
    max_val = np.max(data)
    return 2 * (data - min_val) / (max_val - min_val) - 1

lead_I_real = np.array([min_max_normalize(ecg) for ecg in lead_I_real])
lead_I_fake = np.array([min_max_normalize(ecg) for ecg in lead_I_fake])

# ---------------------------------------------------
# Prepare Data for Anomaly Detection with Autoencoder
# ---------------------------------------------------
# Training on normal (real) ECG signals.
# Label convention: 0 = Normal (real), 1 = Anomaly (fake)

X_train = lead_I_real.copy()  # Train only on normal data
X_test = np.concatenate((lead_I_fake, lead_I_real), axis=0)
y_test = np.concatenate((np.ones(lead_I_fake.shape[0]), np.zeros(lead_I_real.shape[0])), axis=0)

# Reshape to (samples, time_steps, features)
X_train = X_train.reshape(-1, X_train.shape[1], 1)
X_test = X_test.reshape(-1, X_test.shape[1], 1)

print(f"X_train shape (normal data): {X_train.shape}")
print(f"X_test shape (combined data): {X_test.shape}")

# -------------------------------
# LSTM Autoencoder Architecture
# -------------------------------
INPUT_SHAPE = (X_train.shape[1], 1)

autoencoder = Sequential()

# Encoder
autoencoder.add(LSTM(128, activation='relu', return_sequences=True, input_shape=INPUT_SHAPE))
autoencoder.add(Dropout(0.2))
autoencoder.add(LSTM(64, activation='relu', return_sequences=False))
# Decoder
autoencoder.add(RepeatVector(INPUT_SHAPE[0]))
autoencoder.add(LSTM(64, activation='relu', return_sequences=True))
autoencoder.add(Dropout(0.2))
autoencoder.add(LSTM(128, activation='relu', return_sequences=True))
# TimeDistributed Dense: using tanh to match normalized data range
autoencoder.add(TimeDistributed(Dense(1, activation='tanh')))

optimizer = Adam(learning_rate=0.001)
autoencoder.compile(optimizer=optimizer, loss='mae')

autoencoder.summary()

# -----------------------------------
# Training the LSTM Autoencoder Model
# -----------------------------------
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

history = autoencoder.fit(X_train, X_train, 
                          epochs=10, 
                          batch_size=32,
                          validation_split=0.1, 
                          callbacks=[early_stop])

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss (MAE)')
plt.legend()
plt.show()

# ------------------------------------------
# Anomaly Detection: Reconstruction Error
# ------------------------------------------
# Compute reconstruction error on training (normal) data to set a threshold
train_reconstructions = autoencoder.predict(X_train)
train_loss = np.mean(np.abs(train_reconstructions - X_train), axis=1)
threshold = np.mean(train_loss) + 3 * np.std(train_loss)
print("Reconstruction error threshold: {:.4f}".format(threshold))

# Compute reconstruction error on test data
reconstructions = autoencoder.predict(X_test)
test_loss = np.mean(np.abs(reconstructions - X_test), axis=1)

# Flag anomalies where error exceeds threshold
y_pred = (test_loss > threshold).astype(int)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real (Normal)", "Fake (Anomaly)"])
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap='viridis')
plt.title("Confusion Matrix for Anomaly Detection")
plt.show()

# ------------------------------------------
# Visualizing Anomalies (Fake ECG Signals)
# ------------------------------------------
anomaly_indices = np.where((y_test == 1) & (y_pred == 1))[0]
if len(anomaly_indices) > 0:
    idx = anomaly_indices[0]  # Visualize the first detected anomaly
    original_signal = X_test[idx].squeeze()
    reconstructed_signal = reconstructions[idx].squeeze()
    error = np.abs(original_signal - reconstructed_signal)
    
    plt.figure(figsize=(12, 6))
    plt.plot(original_signal, label="Original Signal", color='blue')
    plt.plot(reconstructed_signal, label="Reconstructed Signal", color='red', linestyle='--')
    plt.fill_between(range(len(original_signal)), original_signal, reconstructed_signal, 
                     color='gray', alpha=0.3, label="Reconstruction Error")
    plt.title(f"Anomaly Visualization\nMean Reconstruction Error: {np.mean(error):.4f}")
    plt.xlabel("Time Steps")
    plt.ylabel("Normalized Amplitude")
    plt.legend()
    plt.show()
else:
    print("No anomalies detected among the fake signals.")


In [ ]:
# --------------------------
# Save the Autoencoder Model
# --------------------------
autoencoder.save("lstm_autoencoder_ecg.h5")
print("LSTM Autoencoder model saved successfully.")